# Phase 1: Genomic Cohort Construction & Metadata ETL

### **Project**: PyGDC-RNA-ETL  
#### **Description**: This notebook provides an interface to inspect, filter, and architect GDC cohorts by integrating transcriptomic data with somatic mutation profiles and clinical metadata.
---
#### Core Functionality
This module automates the identification of patient cohorts that meet specific molecular and clinical criteria, standardizing metadata for downstream Machine Learning or Differential Expression analysis.

#### Workflow
1. **Mutation Discovery**: Identify samples with specific somatic mutations via the `/ssm_occurrences` endpoint.
2. **RNA-seq Metadata Extraction**: Query and verify the availability of RNA-seq data via the `/files` endpoint.
3. **Feature Alignment**: Intersect both datasets by `case_id`.
4. **Cohort Profile and QC**: Inspect clinical composition and mutation distribution prior to bulk download.

> **Note**: This pipeline is compatible with all projects hosted on the NCI Genomic Data Commons (GDC). Simply update the `PROJECT_ID` and desired `MUTATED_GENES` in the configuration section to adapt the cohort builder to any cancer type or clinical dataset.

## 0. Dependencies

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries successfully imported")

## 1. Configure cohort

For reference, the following table summarizes the customizable parameters:

| GDC API Field | Description | Typical Values |
| :--- | :--- | :--- |
| `cases.samples.sample_type` | Sample type | Primary Tumor, Solid Tissue Normal, Recurrent Tumor |
| `ssm.consequence.transcript.gene.symbol` | Somatic mutation in selected genes | <details><summary>*View examples*</summary> <ul><li>TP53</li><li>TP53 and EGFR</li><li>EGFR or KRAS</li></details>| |
| `cases.tobacco_smoking_status` | Smoking Status | <details><summary>*View all 6 options*</summary> <ul><li>Lifelong Non-Smoker</li><li>Current Smoker</li><li>Current Reformed Smoker for > 15 yrs</li><li>Current Reformed Smoker for < or = 15 yrs</li><li>Current Reformed Smoker, Duration Not Specified</li><li>Not Reported</li></details>|
| `cases.diagnoses.ajcc_pathologic_stage` | Clinical stage | Stage I, II, III, IV |
| `cases.demographic.vital_status` | Vital status | Alive, Dead |
| `cases.demographic.gender` | Sex at birth | male, female |
| `cases.diagnoses.age_at_diagnosis` | Age at diagnosis (in days) | *integer* |

In [ ]:
# ── API Configuration ────────────────────────
# Standard GDC API endpoint
GDC_API = "https://api.gdc.cancer.gov"

# ── Project selection ────────────────────────
PROJECT_ID = "TCGA-LUAD"

# ── Sample Type ──────────────────────────────
# Options: "Primary Tumor", "Solid Tissue Normal", "Recurrent Tumor"
# None → No filter
SAMPLE_TYPE = "Primary Tumor"

# ── Somatic Mutations ────────────────────────
## List defining which genes to analyze for mutations.
## It must be a list, not a str, even for single genes. Example:
#   ["EGFR"]            → only EGFR
#   ["EGFR", "KRAS"]    → mutations in EGFR and/or KRAS
#   []                  → no mutation filter
MUTATED_GENES = ["EGFR", "KRAS"]
## Define multiple gene parameter:
#   "any"  → mutations in AT LEAST one gene
#   "all"  → mutations in ALL genes
MUTATED_GENES_MODE = "any"
## Define mutation condition for sample selection:
#   "True"  → select samples with mutations
#   "False" → select samples without mutations
# None → No filter (select all samples)
REQUIRE_MUTATED = None

# ── Tobacco smoking status ────────────────────
# Options:   "Lifelong Non-Smoker"
#            "Current Smoker"
#            "Current Reformed Smoker for > 15 yrs"
#            "Current Reformed Smoker, Duration Not Specified"
#            "Current Reformed Smoker for < or = 15 yrs"
#            "Not Reported"
# Recommendation Ever vs Never:
#    Never, define → ["Lifelong Non-Smoker"]
#    Ever, define  → ["Current Smoker", "Current Reformed Smoker for > 15 yrs", "Current Reformed Smoker, Duration Not Specified", "Current Reformed Smoker for < or = 15 yrs"]
#    None       → No filter
TOBACCO_SMOKING_STATUS = None

# ── Pathologic Stage ─────────────────────────
# Options: "Stage I", "Stage II", "Stage III", "Stage IV"
# None → No filter
CLINICAL_STAGE = ["Stage I", "Stage IV"]

# ── Vital status ──────────────────────────────
# Options: "Alive", "Dead"
# None → No filter
VITAL_STATUS = None

# ── Patient sex at birth ──────────────────────
# Options: "male", "female"
# None → no filter
SEX = None

# ── Age Range at Diagnosis ──────────────
# Range: (min_age, max_age)
# None → no filter
AGE_RANGE = None

# ── Maximum number of samples to download ─────
# None → ALL filtered samples
MAX_SAMPLES = 4000

# ── Reproducibility ───────────────────────────
# Fixed random seed for sample capping, used when MAX_SAMPLES is applied in Section 4.
# Change this value to get a different random subset.
RANDOM_STATE = 13

# Internal mapping for stage expansion.
# The pipeline maps high-level clinical stages (I-IV) to their specific GDC subtypes (A, B, C).
# Modify STAGE_GROUPS if you need to group clinical stages differently.
STAGE_GROUPS = {
        "Stage I": ["Stage I", "Stage IA", "Stage IB", "Stage IC"],
        "Stage II": ["Stage II", "Stage IIA", "Stage IIB", "Stage IIC"],
        "Stage III": ["Stage III", "Stage IIIA", "Stage IIIB", "Stage IIIC"],
        "Stage IV": ["Stage IV", "Stage IVA", "Stage IVB"]
    }

# Validate configuration:

if MUTATED_GENES_MODE not in ("any", "all"):
    raise ValueError(f"MUTATED_GENES_MODE must be 'any' or 'all'. Got: '{MUTATED_GENES_MODE}'")

if VITAL_STATUS not in ("Alive", "Dead", None):
    raise ValueError(f"VITAL_STATUS must be 'Alive', 'Dead' or None. Got: '{VITAL_STATUS}'")

if SEX not in ("male", "female", None):
    raise ValueError(f"SEX must be 'male', 'female' or None. Got: '{SEX}'")

if REQUIRE_MUTATED not in (True, False, None):
    raise ValueError(f"REQUIRE_MUTATED must be True, False or None. Got: '{REQUIRE_MUTATED}'")

if AGE_RANGE is not None:
    if not (isinstance(AGE_RANGE, (tuple, list)) and len(AGE_RANGE) == 2 and AGE_RANGE[0] <= AGE_RANGE[1]):
        raise ValueError(f"AGE_RANGE must be a (min_age, max_age) tuple with min <= max. Got: {AGE_RANGE}")
    
if MAX_SAMPLES is not None and (not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES <= 0):
    raise ValueError(f"MAX_SAMPLES must be a positive integer or None. Got: '{MAX_SAMPLES}'")

print("Configuration is valid")

## 2. Get samples with Somatic Mutations

This query goes to `/ssm_occurrences` (somatic mutations) endpoint, and recovers all `case_id` from the selected project harboring mutations in the selected gene/genes.

>**Note**: Mutations are identified specifically within `Primary Tumor` samples, since somatic mutation calling in GDC is most robust for primary sites, we use this as the standard to label the patient's mutational status.

In [ ]:
def get_mutated_cases(project_id: str, genes: list, mode: str = "any"):
    """
    Returns case_ids with somatic mutations in selected genes.

    Parameters
    ----------
    project_id: str
        The selected project identifier, e.g. "TCGA-LUAD"
    genes : list[str]
        List of gene symbols, e.g. ["EGFR", "KRAS"]
    mode : "any" | "all"
        "any" → cases with mutations in AT LEAST one gene
        "all" → cases with mutations in ALL genes (intersection)
    """
    if not genes:
        return set(), {}

    def query_one_gene(gene):
        filters = {
            "op": "and",
            "content": [
                {"op": "=",
                 "content": {"field": "case.project.project_id", "value": project_id}},
                {"op": "=",
                 "content": {"field": "ssm.consequence.transcript.gene.symbol", "value": gene}},
                {"op": "=", "content": {"field": "case.samples.sample_type", "value": "Primary Tumor"}}
            ]
        }
        params = {
            "filters": json.dumps(filters),
            "fields":  "case.case_id,case.submitter_id",
            "format":  "JSON",
            "size":    "5000"
        }
        r = requests.get(f"{GDC_API}/ssm_occurrences", params=params, timeout=60)
        r.raise_for_status()
        try:
            response_data = r.json()["data"]
        except (KeyError, ValueError) as e:
            raise RuntimeError(f"Unexpected GDC API response: {e}\nRaw response: {r.text[:300]}")
        hits = response_data["hits"]
        total_available = response_data["pagination"]["total"]
        
        # Warning if params size is a limiting factor
        if total_available > int(params["size"]):
            print(f"\n    [WARNING] Hit limit ({params['size']}) reached. "
                  f"Found {total_available} files → {total_available - int(params['size'])} cases are missing.")
        
        ids        = {h["case"]["case_id"] for h in hits}
        return ids

    # One query per gene and then combine
    results = {}
    for gene in genes:
        print(f"  Checking mutations in {gene}...", end=" ")
        ids = query_one_gene(gene)
        results[gene] = ids
        print(f"{len(ids)} cases")

    # Combine based on the selected mode
    if mode == "any":
        combined_ids = set.union(*results.values())
    elif mode == "all":
        combined_ids = set.intersection(*results.values())
    else:
        raise ValueError("mode must be 'any' or 'all'")

    # combined_ids → final cohort
    # results      → dictionary detailing gene mutations to case_ids
    return combined_ids, results


print(f"Checking mutations {MUTATED_GENES} (mode: {MUTATED_GENES_MODE})...")
mutated_case_ids, per_gene_ids = get_mutated_cases(
    PROJECT_ID, MUTATED_GENES, mode=MUTATED_GENES_MODE
)
print(f"  Total cases with mutation found: {len(mutated_case_ids)}")

## 3. Query RNA-seq files

This section defines two functions:
- `build_file_filters` constructs the API filter object from the configured parameters. Kept separate so the filter logic can be inspected or modified independently.
- `query_rnaseq_files` then submits the request and parses the response into a metadata DataFrame. Mutation status is NOT used as a filter here; it will be applied as a label in the next step.

In [ ]:
def build_file_filters(project_id, sample_type, clinical_stage,
                       vital_status, tobacco_smoking_status, sex, age_range, stage_map=None):
    """Build the filter object for the /files endpoint."""
    conditions = [
        # Define project identifier
        {"op": "=",
         "content": {"field": "cases.project.project_id", "value": project_id}},
        # Define data type (i.e. gene expression)
        {"op": "=",
         "content": {"field": "data_type", "value": "Gene Expression Quantification"}},
        # Define method for quantification
        {"op": "=",
         "content": {"field": "analysis.workflow_type", "value": "STAR - Counts"}},
        # Get only open, unprotected data
        {"op": "=",
         "content": {"field": "access", "value": "open"}},
    ]

    if sample_type is not None:
        conditions.append(
            {"op": "=",
            "content": {"field": "cases.samples.sample_type", "value": sample_type}}
        )
    
    if clinical_stage is not None and stage_map is not None:
        input_stages = clinical_stage if isinstance(clinical_stage, list) else [clinical_stage]
        expanded_stages = []
        for s in input_stages:
            # Expand using the map if the key exists, else use the raw string
            expanded_stages.extend(stage_map.get(s, [s]))
        
        val = list(set(expanded_stages))
        conditions.append(
            {"op": "in",
            "content": {"field": "cases.diagnoses.ajcc_pathologic_stage", "value": val}}
        )   

    if vital_status is not None:
        conditions.append(
            {"op": "=",
             "content": {"field": "cases.demographic.vital_status", "value": vital_status}}
        )

    if tobacco_smoking_status is not None:
        val = tobacco_smoking_status if isinstance(tobacco_smoking_status, list) else [tobacco_smoking_status]
        conditions.append(
            {"op": "in",
             "content": {"field": "cases.exposures.tobacco_smoking_status", "value": val}}
        )

    if sex is not None:
        conditions.append(
            {"op": "=",
             "content": {"field": "cases.demographic.gender", "value": sex}}
        )

    if age_range is not None:
        conditions.append(
            {"op": ">=",
             "content": {"field": "cases.diagnoses.age_at_diagnosis",
                         "value": age_range[0] * 365.25}}
        )
        conditions.append(
            {"op": "<=",
             "content": {"field": "cases.diagnoses.age_at_diagnosis",
                         "value": age_range[1] * 365.25}}
        )

    return {"op": "and", "content": conditions}


def query_rnaseq_files(filters):
    """Return a DataFrame with metadata for all files matching the filters."""
    fields = [
        "file_id",
        "file_name",
        "file_size",
        "cases.case_id",
        "cases.submitter_id",
        "cases.samples.sample_type",
        "cases.demographic.gender",
        "cases.demographic.vital_status",
        "cases.demographic.days_to_death",
        "cases.diagnoses.ajcc_pathologic_stage",
        "cases.diagnoses.age_at_diagnosis",
        "cases.exposures.tobacco_smoking_status",
    ]
    params = {
        "filters": json.dumps(filters),
        "fields":  ",".join(fields),
        "format":  "JSON",
        "size":    "5000"
    }
    r = requests.get(f"{GDC_API}/files", params=params, timeout=60)
    r.raise_for_status()
    try:
        response_data = r.json()["data"]
    except (KeyError, ValueError) as e:
        raise RuntimeError(f"Unexpected GDC API response: {e}\nRaw response: {r.text[:300]}")
    hits = response_data["hits"]
    total_available = response_data["pagination"]["total"]

    # Warning if params size is a limiting factor
    if total_available > int(params["size"]):
        print(f"\n    [WARNING] Hit limit ({params['size']}) reached. "
              f"Found {total_available} files → {total_available - int(params['size'])} cases are missing.")
    
    rows = []
    for f in hits:
        
        # Warn if a file is associated with more than one case
        if len(f.get("cases", [])) > 1:
            print(f"  [WARNING] file {f['file_id']} has "
                  f"{len(f['cases'])} case associations → "
                  f"only the first will be used.")

        # Warn if a file is not associated with a case
        if not f.get("cases"):
            print(f"  [WARNING] file {f['file_id']} has no case associations → skipped.")
            continue
        
        case   = f["cases"][0]
        diag   = case.get("diagnoses", [{}])[0]
        demo   = case.get("demographic", {})
        sample = case.get("samples", [{}])[0]
        expose = case.get("exposures", [{}])[0] if case.get("exposures") else {}

        age_days = diag.get("age_at_diagnosis")
        age_yr   = round(age_days / 365.25, 1) if age_days else None

        rows.append({
            "file_id":             f["file_id"],
            "file_name":           f["file_name"],
            "file_size_mb":        round(f["file_size"] / 1e6, 2),
            "case_id":             case["case_id"],
            "submitter_id":        case["submitter_id"],
            "sample_type":         sample.get("sample_type"),
            "gender":              demo.get("gender"),
            "vital_status":        demo.get("vital_status"),
            "days_to_death":       demo.get("days_to_death"),
            "stage":               diag.get("ajcc_pathologic_stage"),
            "age_at_diagnosis_yr": age_yr,
            "tobacco_smoking":     expose.get("tobacco_smoking_status"),
        })
        
    return pd.DataFrame(rows)


print("Building file filters...")
filters = build_file_filters(
    project_id=PROJECT_ID,
    sample_type=SAMPLE_TYPE,
    clinical_stage=CLINICAL_STAGE,
    vital_status=VITAL_STATUS,
    tobacco_smoking_status=TOBACCO_SMOKING_STATUS,
    sex=SEX,
    age_range=AGE_RANGE,
    stage_map=STAGE_GROUPS
)
print("Querying RNA-seq files...")
df_all = query_rnaseq_files(filters)

if df_all.empty:
    print("\n [ERROR]: No RNA-seq files found matching these filters.")
else:
    print(f"Files found: {len(df_all)}")

## 4. Assign mutation labels and inspect cohort

Mutation status is added as columns to the metadata DataFrame — one binary column per gene (e.g. `mut_EGFR`, `mut_KRAS`) plus a combined column (`mutated_any` or `mutated_all`) following the `MUTATED_GENES_MODE` setting.

The cohort is then optionally filtered by mutation status (`REQUIRE_MUTATED`) and capped to a maximum number of samples (`MAX_SAMPLES`) with a fixed random seed for reproducibility.

Finally, a cohort summary is printed to inspect:
- **Mutation label balance** — mutant vs wild-type counts per gene
- **Demographic distributions** — stage, vital status, sex, and smoking history

> **Note**: If mutant and wild-type groups differ significantly in stage, sex, or smoking status, those variables may act as **confounders** in downstream applications and should be accounted for during modelling.

In [ ]:
# ── Assign per-gene mutation columns ──────────────────────────
if per_gene_ids:
    for gene, ids in per_gene_ids.items():
        df_all[f"mut_{gene}"] = df_all["case_id"].isin(ids)

# Combined mutation column (follows MUTATED_GENES_MODE logic)
mut_col = f"mutated_{MUTATED_GENES_MODE}"
if mutated_case_ids:
    df_all[mut_col] = df_all["case_id"].isin(mutated_case_ids)
else:
    df_all[mut_col] = False

# ── Apply REQUIRE_MUTATED filter ──────────────────────────────
if REQUIRE_MUTATED is True:
    df_selected = df_all[df_all[mut_col]].copy()
elif REQUIRE_MUTATED is False:
    df_selected = df_all[~df_all[mut_col]].copy()
else:
    df_selected = df_all.copy()    # all samples, mutation status used as label

# ── Optional sample cap ───────────────────────────────────────
## If sample cap is activated, performs random sampling.
## RANDOM_STATE is defined in Section 1 for reproducibility.
if MAX_SAMPLES is not None:
    df_selected = df_selected.sample(
        min(MAX_SAMPLES, len(df_selected)), random_state=RANDOM_STATE
    )

# ── Cohort summary ────────────────────────────────────────────
print("=" * 60)
print("  COHORT SUMMARY")
print("=" * 60)
print(f"  Total samples:              {len(df_selected)}")
print(f"  Unique cases (patients):    {df_selected['case_id'].nunique()}")
print(f"  Estimated download size:    {df_selected['file_size_mb'].sum():.0f} MB")

print(f"\n  --- Mutation labels ---")
for gene in MUTATED_GENES:
    col = f"mut_{gene}"
    if col in df_selected.columns:
        n_mut = df_selected[col].sum()
        n_wt  = (~df_selected[col]).sum()
        print(f"  {gene}: {n_mut} mutant  |  {n_wt} wild-type")
if MUTATED_GENES:
    n_comb = df_selected[mut_col].sum()
    print(f"  Combined ({MUTATED_GENES_MODE}): {n_comb} mutant  |  {len(df_selected) - n_comb} wild-type")

print(f"\n  --- Demographic distributions ---")
print("\n  Stage:")
print(df_selected["stage"].value_counts().to_string())
print("\n  Vital status:")
print(df_selected["vital_status"].value_counts().to_string())
print("\n  Gender at birth:")
print(df_selected["gender"].value_counts().to_string())
print("\n  Summary of Smoking History (Ever vs Never):")
ever_never_map = {
    "Lifelong Non-Smoker": "Never",
    "Current Smoker": "Ever",
    "Current Reformed Smoker for > 15 yrs": "Ever",
    "Current Reformed Smoker, Duration Not Specified": "Ever",
    "Current Reformed Smoker for < or = 15 yrs": "Ever",
    "Not Reported": "Unknown"
}
smoking_summary = df_selected["tobacco_smoking"].map(ever_never_map).fillna("Unknown")
print(smoking_summary.value_counts().to_string())


plt.figure(figsize=(10, 4))
sns.countplot(data=df_selected, x='stage', hue=f'mutated_{MUTATED_GENES_MODE}')
plt.title("Cohort Composition: Stage vs Mutation Status")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10, 5))
sns.countplot(data=df_selected, x='sample_type', hue=mut_col)
plt.title("Mutation Labeling by Sample Type")
plt.show()

## 5. Save metadata

Saves the annotated cohort metadata to `cohort_metadata.csv`. This file is the input for the download step (`02_download_counts.ipynb`) and contains file IDs, clinical variables, and mutation labels.

In [ ]:
df_selected.to_csv("cohort_metadata.csv", index=False)
print(f"Metadata saved → cohort_metadata.csv ({len(df_selected)} rows)")
print()
print("Available columns (features):")
print(df_selected.columns.tolist())